# DRUG-TARGET AFFINITY PREDICTION — CONCEPT BOTTLENECK MODEL
### SMILES → SELFIES → ESM2 + SELFormer → CBM → pIC50

## ✅ CELL 1 — Install Dependencies

In [ ]:
!pip install selfies fair-esm transformers torch pandas scikit-learn scipy matplotlib seaborn biopython rdkit tqdm huggingface_hub -q
print('Dependencies installed.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.1/93.1 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.1/37.1 MB 16.6 MB/s eta 0:00:00
Dependencies installed.


## ✅ CELL 2 — Mount Google Drive (REQUIRED — keeps files between sessions)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/DTA_CBM'
os.makedirs(f'{DRIVE_ROOT}/outputs',      exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/saved_models', exist_ok=True)
print('Drive mounted. Project folder:', DRIVE_ROOT)

Mounted at /content/drive
Drive mounted. Project folder: /content/drive/MyDrive/DTA_CBM


## ✅ CELL 3 — SMILES → SELFIES (run once, skip if already done)

In [ ]:
from pathlib import Path
SELFIES_OUT = f'{DRIVE_ROOT}/smiles_to_selfies_output.csv'

if Path(SELFIES_OUT).exists():
    print('✅ SELFIES file already exists on Drive — skipping conversion.')
else:
    import pandas as pd, selfies as sf
    from google.colab import files
    uploaded = files.upload()   # upload lassapic.csv

    def smiles_to_selfies(s):
        try: return sf.encoder(s)
        except: return None

    df = pd.read_csv('lassapic.csv')
    df['SELFIES'] = df['Smiles'].apply(smiles_to_selfies)
    df.to_csv(SELFIES_OUT, index=False)
    print(f'Saved {len(df)} rows, valid SELFIES: {df["SELFIES"].notna().sum()}')

✅ SELFIES file already exists on Drive — skipping conversion.


## ✅ CELL 4 — Protein Embedding via ESM-2 (run once)

In [ ]:
PROT_EMB_OUT = f'{DRIVE_ROOT}/protein_embedding.csv'

if Path(PROT_EMB_OUT).exists():
    print('✅ Protein embedding already exists on Drive — skipping.')
else:
    import esm, torch, pandas as pd
    model_esm, alphabet = esm.pretrained.esm2_t6_8M_UR50D()
    batch_converter = alphabet.get_batch_converter()
    model_esm.eval()
    print('ESM-2 hidden size:', model_esm.embed_dim)  # 320

    sequence = 'GTFTWTLSDSEGKDTPGGYCLTRWMLIEAELKCFGNTAVAKCNEKHDEEFCDMLRLFDFNKQAIQRLKAEAQMSIQLINKAVNALINDQLIMKNHLRDIMGIPYCNYSKYWYLNHTTTGRTSLPKCWLVSNGSYLNETHFSDDIEQQADNMITEMLQKEYMERQGKTPLGLVDLFVFSTSFYLISIFLHLVKIPTHRHIVGKSCPKPHRLNHMGICSCGLYKQPGVPVKWKR'
    labels, strs, tokens = batch_converter([('protein', sequence)])
    with torch.no_grad():
        results = model_esm(tokens, repr_layers=[6])
    embedding = results['representations'][6][:, 1:-1].mean(1)
    print('Protein Embedding shape:', embedding.shape)  # (1, 320)
    pd.DataFrame(embedding.numpy()).to_csv(PROT_EMB_OUT, index=False)
    print('Saved protein_embedding.csv to Drive.')

✅ Protein embedding already exists on Drive — skipping.


## ✅ CELL 5 — Drug Embedding via SELFormer (run once)

In [ ]:
DRUG_EMB_OUT = f'{DRIVE_ROOT}/selformer_embeddings_final.csv'

if Path(DRUG_EMB_OUT).exists():
    print('✅ Drug embeddings already exist on Drive — skipping.')
else:
    from transformers import AutoTokenizer, AutoModel
    import torch, pandas as pd, numpy as np

    tok_sel   = AutoTokenizer.from_pretrained('HUBioDataLab/SELFormer')
    model_sel = AutoModel.from_pretrained('HUBioDataLab/SELFormer')
    model_sel.eval()
    print('SELFormer hidden size:', model_sel.config.hidden_size)  # 768

    df = pd.read_csv(SELFIES_OUT)
    df = df[df['SELFIES'].notna()].reset_index(drop=True)

    def get_drug_emb(s):
        try:
            inp = tok_sel([s], return_tensors='pt', truncation=True, padding=True, max_length=512)
            with torch.no_grad():
                out = model_sel(**inp)
            return out.last_hidden_state.mean(dim=1).squeeze().numpy()
        except: return None

    embs       = df['SELFIES'].apply(get_drug_emb)
    emb_matrix = np.vstack(embs.values)
    emb_df     = pd.DataFrame(emb_matrix, columns=[f'emb_{i}' for i in range(emb_matrix.shape[1])])
    pd.concat([df, emb_df], axis=1).to_csv(DRUG_EMB_OUT, index=False)
    print(f'Drug embeddings shape: {emb_matrix.shape} — saved to Drive.')

✅ Drug embeddings already exist on Drive — skipping.


## ✅ CELL 6 — Imports & Global Setup

In [ ]:
import os, json, logging
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Optional
from dataclasses import dataclass, field
import warnings
warnings.filterwarnings('ignore')

import selfies as sf
from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski, QED

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from transformers import AutoTokenizer, AutoModel

from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from scipy.stats import pearsonr, spearmanr

import matplotlib
matplotlib.use('Agg')   # headless-safe; plt.show() still works in Colab
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import tqdm

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)s %(message)s',
    handlers=[logging.StreamHandler()]
)
logger = logging.getLogger(__name__)

def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
print('All imports OK.')

Device: cpu
All imports OK.


## ✅ CELL 7 — Configuration (paths point to Google Drive)

In [ ]:
@dataclass
class Config:
    # ── Paths ─────────────────────────────────────────────────────────────
    data_path:      str = '/content/drive/MyDrive/DTA_CBM/lassapic_with_pic50.csv'  # merged file with pIC50
    output_dir:     str = '/content/drive/MyDrive/DTA_CBM/outputs'
    model_save_dir: str = '/content/drive/MyDrive/DTA_CBM/saved_models'

    # ── Dimensions ────────────────────────────────────────────────────────
    drug_dim:    int = 768
    protein_dim: int = 320
    fused_dim:   int = 1088
    num_concepts: int = 12

    # ── Model IDs ─────────────────────────────────────────────────────────
    selformer_model: str = 'HUBioDataLab/SELFormer'
    esm2_model:      str = 'facebook/esm2_t6_8M_UR50D'

    max_drug_length:    int = 128
    max_protein_length: int = 512           # FIX ① reduced from 1024 → saves memory

    # ── Training ──────────────────────────────────────────────────────────
    batch_size:          int   = 16         # FIX ② doubled
    learning_rate:       float = 2e-5       # FIX ③ was 1e-4 → transformer LR must be low
    head_lr:             float = 1e-4       # FIX ③ separate LR for CBM/predictor head
    weight_decay:        float = 1e-4
    num_epochs:          int   = 50         # 50 epochs ≈ 3-4 hrs on Colab GPU
    dropout_rate:        float = 0.2        # FIX ⑤ reduced from 0.3
    concept_loss_weight: float = 0.2        # FIX ⑥ reduced from 0.3
    save_interval:       int   = 10
    test_split:          float = 0.2
    val_split:           float = 0.1

    # ── pIC50 normalisation (filled in Cell 12 from training data) ────────
    # FIX ⑦ — THE ROOT CAUSE OF R²=0: model output starts near 0 but raw
    # pIC50 lives in [4, 11]. Without z-score, MSE is huge and the model
    # collapses to predicting the training mean (R² = 0).
    pic50_mean: float = 6.0                 # placeholder; overwritten in Cell 12
    pic50_std:  float = 1.5                 # placeholder; overwritten in Cell 12

    # ── Concepts ──────────────────────────────────────────────────────────
    binary_concepts:     List[int] = field(default_factory=lambda: [3,4,5,6,7])
    continuous_concepts: List[int] = field(default_factory=lambda: [0,1,2,8,9,10,11])
    concept_names: List[str] = field(default_factory=lambda: [
        'bioavailability','metabolic_stability','binding_compatibility',
        'zinc_binding','hydrogen_bond_donor','hydrogen_bond_acceptor',
        'hydrophobic_contact','steric_clash',
        'molecular_weight_norm','logP_norm','rotatable_bonds_norm','aromatic_rings_norm'
    ])

    def create_dirs(self):
        Path(self.output_dir).mkdir(parents=True, exist_ok=True)
        Path(self.model_save_dir).mkdir(parents=True, exist_ok=True)

config = Config()
config.create_dirs()

BEST_MODEL = Path(config.model_save_dir) / 'best_model.pt'
HIST_FILE  = Path(config.output_dir)     / 'training_history.json'
PROC_DATA  = Path(config.output_dir)     / 'processed_data.csv'

print(f'drug_dim={config.drug_dim}  protein_dim={config.protein_dim}  fused_dim={config.fused_dim}')
print(f'best_model.pt exists : {BEST_MODEL.exists()}')
print(f'history.json  exists : {HIST_FILE.exists()}')
print(f'processed_data exists: {PROC_DATA.exists()}')


drug_dim=768  protein_dim=320  fused_dim=1088
best_model.pt exists : False
history.json  exists : True
processed_data exists: False


## ✅ CELL 8 — Data Processor

In [ ]:
_DEFAULT_PROTEIN_SEQ = (
    "MGCGCSSHPEDDWMENIDVCENCHYPIVPLEYDPSTVSSIFQQRLENDYINKLIHNIVDQNEG"
    "TLKKLIESARIQGPDDQRLPGEISALVKDLSQLKEHMKKDTPALQIVDAHEHLEIRRQRQKL"
    "DERIRKQMEAQNRAEGRKPPVLTPKPLRRMEAFACTSEGDRDMHILNKSTKTL"
)

class DataProcessor:
    def __init__(self, config, activity_col=None):
        self.config       = config
        self.cnames       = config.concept_names
        self.activity_col = activity_col   # injected by Cell 12

    def smiles_to_selfies(self, smiles):
        try:    return sf.encoder(smiles)
        except: return None

    def _get_smiles(self, row):
        for c in ['Smiles','smiles','SMILES']:
            if c in row.index and pd.notna(row[c]): return row[c]
        raise ValueError('No SMILES column found')

    def _get_seq(self, row):
        for c in ['sequence','protein_sequence']:
            if c in row.index and pd.notna(row[c]): return row[c]
        return _DEFAULT_PROTEIN_SEQ

    def calc_drug_concepts(self, smiles):
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return {n: 0.0 for n in self.cnames}
        mw   = Descriptors.MolWt(mol)
        logp = Descriptors.MolLogP(mol)
        hbd  = Lipinski.NumHDonors(mol)
        hba  = Lipinski.NumHAcceptors(mol)
        rotb = Descriptors.NumRotatableBonds(mol)
        arom = Lipinski.NumAromaticRings(mol)
        qed  = QED.qed(mol)
        violations = sum([mw > 500, logp > 5, hbd > 5, hba > 10])
        met_stab   = max(0.0, 1.0 - violations / 4.0)
        bind_compat = float(qed)
        steric      = 1.0 if (mw > 500 or rotb > 10) else 0.0
        return {
            'bioavailability':        float(qed),
            'metabolic_stability':    float(met_stab),
            'binding_compatibility':  float(bind_compat),
            'zinc_binding':           0.0,
            'hydrogen_bond_donor':    1.0 if hbd > 2 else 0.0,
            'hydrogen_bond_acceptor': 1.0 if hba > 3 else 0.0,
            'hydrophobic_contact':    1.0 if logp > 2 else 0.0,
            'steric_clash':           float(steric),
            'molecular_weight_norm':  float(np.clip(mw  / 600.0,       0, 1)),
            'logP_norm':              float(np.clip((logp + 5) / 15.0,  0, 1)),
            'rotatable_bonds_norm':   float(np.clip(rotb / 15.0,        0, 1)),
            'aromatic_rings_norm':    float(np.clip(arom / 6.0,         0, 1)),
        }

    def calc_protein_concepts(self, seq):
        zinc = any(m in seq.upper() for m in ['CCHC','CCHH','CCCC','HXXH','CXXC'])
        return {'zinc_binding': 1.0 if zinc else 0.0, 'steric_clash': 0.0,
                'binding_compatibility': 0.5, 'metabolic_stability': 0.5}

    def _resolve_pic50(self, df):
        """Convert whatever activity column exists → pIC50."""
        col = self.activity_col
        vals = df[col].clip(lower=1e-3)

        # Decide whether column is already pIC50 or raw IC50/Ki/Kd
        median_val = vals.median()
        if median_val < 50:
            # Looks like it is already a pIC50 / log-scale value
            logger.info(f"Column '{col}' median={median_val:.2f} → treated as pIC50 directly.")
            return vals.astype(float)
        else:
            # Assume nanomolar IC50/Ki/Kd → convert
            logger.info(f"Column '{col}' median={median_val:.1f} → converting nM IC50 to pIC50.")
            return -np.log10(vals * 1e-9)

    def load_and_process(self, filepath):
        df = pd.read_csv(filepath)
        logger.info(f'Loaded {len(df)} rows | cols={list(df.columns)}')

        # ── Resolve activity → pIC50 ────────────────────────────────────
        if 'pIC50' in df.columns:
            df['pIC50'] = df['pIC50'].astype(float)
        elif self.activity_col and self.activity_col in df.columns:
            df['pIC50'] = self._resolve_pic50(df)
        else:
            raise ValueError(
                f"Cannot find activity column.\nColumns: {list(df.columns)}\n"
                "Set ACTIVITY_COL in Cell 12."
            )

        bad = ((df['pIC50'] < 2) | (df['pIC50'] > 14)).sum()
        if bad > 0:
            logger.warning(f'⚠️  {bad} pIC50 values outside [2,14] — check IC50 units')
        df = df[df['pIC50'].between(2, 14)].copy()
        logger.info(f'pIC50 range after filter: {df["pIC50"].min():.2f}–{df["pIC50"].max():.2f}')

        if 'sequence' not in df.columns and 'protein_sequence' not in df.columns:
            logger.warning('No protein sequence column → using default Lck sequence.')
            df['sequence'] = _DEFAULT_PROTEIN_SEQ

        sc = 'SELFIES' if 'SELFIES' in df.columns else ('selfies' if 'selfies' in df.columns else None)
        if sc:
            df = df[df[sc].notna()].copy()
            df['selfies'] = df[sc]
        else:
            res, idxs = [], []
            for idx, row in tqdm.tqdm(df.iterrows(), total=len(df)):
                s = self.smiles_to_selfies(self._get_smiles(row))
                if s: res.append(s); idxs.append(idx)
            df = df.loc[idxs].copy(); df['selfies'] = res

        cdata = []
        for _, row in tqdm.tqdm(df.iterrows(), total=len(df), desc='Concepts'):
            d = self.calc_drug_concepts(self._get_smiles(row))
            p = self.calc_protein_concepts(self._get_seq(row))
            merged = {**d, **{k: v for k, v in p.items()
                              if k in ['zinc_binding', 'steric_clash']}}
            cdata.append(merged)
        cdf = pd.DataFrame(cdata)
        for col in cdf.columns: df[col] = cdf[col].values
        logger.info(f'Processed shape: {df.shape}')
        return df

print('DataProcessor defined.')


DataProcessor defined.


## ✅ CELL 9 — Model Architecture

In [ ]:
class SELFormerEncoder(nn.Module):
    def __init__(self, model_name, max_length=128):
        super().__init__()
        self.max_length = max_length
        self.tokenizer  = AutoTokenizer.from_pretrained(model_name)
        self.encoder    = AutoModel.from_pretrained(model_name)
        self.encoder.train()
        logger.info(f'SELFormer hidden_size={self.encoder.config.hidden_size}')

    def forward(self, selfies_list):
        inp = self.tokenizer(selfies_list, padding=True, truncation=True,
                             max_length=self.max_length, return_tensors='pt').to(device)
        out  = self.encoder(**inp)
        mask = inp['attention_mask'].unsqueeze(-1)
        return (out.last_hidden_state * mask).sum(1) / mask.sum(1)  # (B,768)


class ESM2Encoder(nn.Module):
    def __init__(self, model_name, max_length=1024):
        super().__init__()
        self.max_length  = max_length
        self.tokenizer   = AutoTokenizer.from_pretrained(model_name)
        self.encoder     = AutoModel.from_pretrained(model_name)
        self.encoder.train()
        self.hidden_size = self.encoder.config.hidden_size  # 320
        logger.info(f'ESM-2 hidden_size={self.hidden_size}')

    def forward(self, seq_list):
        inp = self.tokenizer(seq_list, padding=True, truncation=True,
                             max_length=self.max_length, return_tensors='pt').to(device)
        out  = self.encoder(**inp)
        mask = inp['attention_mask'].unsqueeze(-1)
        return (out.last_hidden_state * mask).sum(1) / mask.sum(1)  # (B,320)


class ConceptBottleneck(nn.Module):
    def __init__(self, input_dim, num_concepts, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim,512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(512,256),       nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256,num_concepts)
        )
        for m in self.modules():
            if isinstance(m,nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None: nn.init.zeros_(m.bias)

    def forward(self, fused, bin_idx, cont_idx):
        logits = self.net(fused)
        out    = torch.zeros_like(logits)
        if bin_idx:  out[:, bin_idx]  = torch.sigmoid(logits[:, bin_idx])
        if cont_idx: out[:, cont_idx] = F.relu(logits[:, cont_idx])
        return out


class AffinityPredictor(nn.Module):
    def __init__(self, num_concepts, dropout):
        super().__init__()
        self.predictor = nn.Sequential(
            nn.Linear(num_concepts,128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128,64),           nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64,1)
        )
        for m in self.modules():
            if isinstance(m,nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None: nn.init.zeros_(m.bias)

    def forward(self, c): return self.predictor(c)


class DTAModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config          = config
        self.drug_enc        = SELFormerEncoder(config.selformer_model, config.max_drug_length)
        self.prot_enc        = ESM2Encoder(config.esm2_model, config.max_protein_length)

        # Auto-correct fused_dim from actual encoder output sizes
        actual_fused = config.drug_dim + self.prot_enc.hidden_size
        if actual_fused != config.fused_dim:
            logger.warning(f'Correcting fused_dim {config.fused_dim}→{actual_fused}')
            config.fused_dim = actual_fused

        self.cbm  = ConceptBottleneck(config.fused_dim, config.num_concepts, config.dropout_rate)
        self.pred = AffinityPredictor(config.num_concepts, config.dropout_rate)
        logger.info(f'DTAModel ready | params={self.n_params():,} | fused_dim={config.fused_dim}')

    def n_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

    def forward(self, selfies_list, seq_list, return_concepts=True):
        d = self.drug_enc(selfies_list)      # (B,768)
        p = self.prot_enc(seq_list)          # (B,320)
        f = torch.cat([d, p], dim=1)         # (B,1088)
        c = self.cbm(f, self.config.binary_concepts, self.config.continuous_concepts)
        y = self.pred(c)                     # (B,1)
        return (y, c) if return_concepts else y

print('Model defined.')

Model defined.


## ✅ CELL 10 — Loss, Dataset, DataLoader

In [ ]:
class CBMLoss(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.cfg = config
        self.mse = nn.MSELoss()
        self.bce = nn.BCELoss()

    def forward(self, pc, tc, py, ty):
        cl = torch.tensor(0.0, device=pc.device)
        if self.cfg.binary_concepts:
            cl = cl + self.bce(
                pc[:, self.cfg.binary_concepts].clamp(1e-6, 1-1e-6),
                tc[:, self.cfg.binary_concepts].clamp(0., 1.)
            )
        if self.cfg.continuous_concepts:
            cl = cl + self.mse(pc[:, self.cfg.continuous_concepts],
                               tc[:, self.cfg.continuous_concepts])
        al = self.mse(py.squeeze(), ty.squeeze())
        total = self.cfg.concept_loss_weight * cl + (1 - self.cfg.concept_loss_weight) * al
        return {'total': total, 'concept': cl, 'affinity': al}


class DTADataset(Dataset):
    def __init__(self, df, config):
        self.df  = df.reset_index(drop=True)
        self.cfg = config

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        r  = self.df.iloc[idx]
        sq = r['sequence'] if 'sequence' in r.index else r['protein_sequence']
        c  = np.array([float(r[n]) if n in r.index else 0.0
                       for n in self.cfg.concept_names], dtype=np.float32)
        # FIX ⑫  use normalised pIC50 for training (raw pIC50 caused R²=0)
        return {'selfies': r['selfies'], 'seq': sq,
                'concepts': torch.FloatTensor(c),
                'pic50':    torch.FloatTensor([float(r['pIC50_norm'])])}


def collate(batch):
    return {'selfies':  [b['selfies']  for b in batch],
            'proteins': [b['seq']      for b in batch],
            'concepts': torch.stack([b['concepts'] for b in batch]),
            'pic50':    torch.stack([b['pic50']    for b in batch])}


def make_loader(ds, bs, shuffle):
    return DataLoader(ds, batch_size=bs, shuffle=shuffle,
                      collate_fn=collate, num_workers=0, pin_memory=False)

print('Loss / Dataset / DataLoader defined.')


Loss / Dataset / DataLoader defined.


## ✅ CELL 11 — Trainer (with history save/load)

In [ ]:
class CBMLoss(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.cfg = config
        self.mse = nn.MSELoss()
        self.bce = nn.BCELoss()

    def forward(self, pc, tc, py, ty):
        cl = torch.tensor(0.0, device=pc.device)
        if self.cfg.binary_concepts:
            cl = cl + self.bce(
                pc[:, self.cfg.binary_concepts].clamp(1e-6, 1-1e-6),
                tc[:, self.cfg.binary_concepts].clamp(0., 1.))
        if self.cfg.continuous_concepts:
            cl = cl + self.mse(pc[:, self.cfg.continuous_concepts],
                               tc[:, self.cfg.continuous_concepts])
        al    = self.mse(py.squeeze(), ty.squeeze())
        total = self.cfg.concept_loss_weight * cl + (1 - self.cfg.concept_loss_weight) * al
        return {'total': total, 'concept': cl, 'affinity': al}


class Trainer:
    def __init__(self, model, config):
        self.model = model
        self.cfg   = config
        self.loss  = CBMLoss(config)

        transformer_params = (list(model.drug_enc.parameters()) +
                              list(model.prot_enc.parameters()))
        head_params        = (list(model.cbm.parameters()) +
                              list(model.pred.parameters()))
        self.opt = AdamW([
            {'params': transformer_params, 'lr': config.learning_rate},
            {'params': head_params,        'lr': config.head_lr},
        ], weight_decay=config.weight_decay)

        self.sch = CosineAnnealingLR(self.opt, T_max=config.num_epochs, eta_min=1e-6)
        self.history = {k: [] for k in
                        ['train_loss','val_loss','val_mse','val_r2','val_pearson','val_spearman']}
        self.best_mse   = float('inf')
        self.best_epoch = -1

    def load_history(self):
        if HIST_FILE.exists():
            with open(HIST_FILE) as f:
                saved = json.load(f)
            self.history = saved.get('history', saved)
            if 'pic50_mean' in saved:
                self.cfg.pic50_mean = saved['pic50_mean']
                self.cfg.pic50_std  = saved['pic50_std']
            logger.info(f'Loaded history ({len(self.history["train_loss"])} epochs)')

    def save_history(self):
        payload = {k: [float(v) for v in vs] for k, vs in self.history.items()}
        payload['pic50_mean'] = float(self.cfg.pic50_mean)
        payload['pic50_std']  = float(self.cfg.pic50_std)
        with open(HIST_FILE, 'w') as f:
            json.dump(payload, f, indent=2)

    def save_model(self, name):
        torch.save({'model_state_dict': self.model.state_dict(),
                    'opt_state_dict':   self.opt.state_dict(),
                    'best_mse':         self.best_mse,
                    'best_epoch':       self.best_epoch,
                    'pic50_mean':       self.cfg.pic50_mean,
                    'pic50_std':        self.cfg.pic50_std},
                   Path(self.cfg.model_save_dir) / name)

    def _run_batch(self, batch, train=True):
        tc = batch['concepts'].to(device)
        ty = batch['pic50'].to(device)
        py, pc = self.model(batch['selfies'], batch['proteins'])
        losses = self.loss(pc, tc, py, ty)
        if train:
            self.opt.zero_grad()
            losses['total'].backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
            self.opt.step()
        return losses['total'].item(), py.detach(), tc.detach(), pc.detach()

    def train_epoch(self, loader, epoch=0, total_epochs=0):
        self.model.train()
        total, n = 0.0, 0
        # ── Single bar that shows epoch context ──────────────────────────
        bar = tqdm.tqdm(loader,
                        desc=f'  TRAIN eval{f" Epoch {epoch}/{total_epochs}" if total_epochs else ""}',
                        leave=False,
                        unit='batch',
                        bar_format='{l_bar}{bar:25}{r_bar}')
        for b in bar:
            l, _, _, _ = self._run_batch(b, train=True)
            total += l; n += 1
            bar.set_postfix({'loss': f'{total/n:.4f}'})
        return total / n

    @torch.no_grad()
    def evaluate(self, loader, epoch=0, total_epochs=0, split='val'):
        self.model.eval()
        total, preds_norm, trues_norm, cps, cts = 0.0, [], [], [], []
        bar = tqdm.tqdm(loader,
                        desc=f'  {split.upper()} eval{f" Epoch {epoch}/{total_epochs}" if total_epochs else ""} ',
                        leave=False,
                        unit='batch',
                        bar_format='{l_bar}{bar:25}{r_bar}')
        for b in bar:
            tc = b['concepts'].to(device)
            ty = b['pic50'].to(device)
            py, pc = self.model(b['selfies'], b['proteins'])
            losses = self.loss(pc, tc, py, ty)
            total += losses['total'].item()
            yp = py.squeeze().cpu().numpy()
            yt = ty.squeeze().cpu().numpy()
            preds_norm.extend([float(yp)] if yp.ndim == 0 else yp.tolist())
            trues_norm.extend([float(yt)] if yt.ndim == 0 else yt.tolist())
            cps.append(pc.cpu().numpy())
            cts.append(tc.cpu().numpy())

        mu, sigma = self.cfg.pic50_mean, self.cfg.pic50_std
        preds = np.array(preds_norm) * sigma + mu
        trues = np.array(trues_norm) * sigma + mu

        mse   = mean_squared_error(trues, preds)
        r2    = r2_score(trues, preds)
        pr, _ = pearsonr(trues,  preds)
        sp, _ = spearmanr(trues, preds)

        cp_arr = np.concatenate(cps, axis=0)
        ct_arr = np.concatenate(cts, axis=0)
        concept_acc = {}
        for idx in self.cfg.binary_concepts:
            pred_b = (cp_arr[:, idx] > 0.5).astype(float)
            concept_acc[self.cfg.concept_names[idx]] = float(
                np.mean(pred_b == ct_arr[:, idx]))

        return {'loss': total/len(loader), 'mse': mse, 'rmse': np.sqrt(mse),
                'r2': r2, 'pearson': pr, 'spearman': sp,
                'preds': preds, 'trues': trues,
                'concept_accuracy': concept_acc}

    def train(self, train_loader, val_loader):
        total = self.cfg.num_epochs
        done  = len(self.history['train_loss'])   # epochs already completed

        # ── Outer epoch bar — always visible, never erased ───────────────
        epoch_bar = tqdm.tqdm(
            range(done + 1, total + 1),
            desc='Overall progress',
            unit='epoch',
            initial=done,
            total=total,
            position=0,
            leave=True,
            bar_format='{l_bar}{bar:30}{r_bar}'
        )

        print(f'\n🚀 Training from epoch {done+1} → {total}  (total={total} epochs)')
        print(f'   Each epoch = {len(train_loader)} train batches + {len(val_loader)} val batches')
        print(f'   DO NOT re-run this cell — it will continue automatically\n')

        for epoch in epoch_bar:
            tl = self.train_epoch(train_loader, epoch, total)
            vm = self.evaluate(val_loader, epoch, total, split='val')
            self.sch.step()

            is_best = vm['mse'] < self.best_mse
            if is_best:
                self.best_mse   = vm['mse']
                self.best_epoch = epoch
                self.save_model('best_model.pt')

            self.history['train_loss'].append(tl)
            self.history['val_loss'].append(vm['loss'])
            self.history['val_mse'].append(vm['mse'])
            self.history['val_r2'].append(vm['r2'])
            self.history['val_pearson'].append(vm['pearson'])
            self.history['val_spearman'].append(vm['spearman'])

            # ── Epoch bar postfix shows live metrics ─────────────────────
            epoch_bar.set_postfix({
                'train':   f'{tl:.4f}',
                'val_mse': f'{vm["mse"]:.4f}',
                'R²':      f'{vm["r2"]:.3f}',
                'pearson': f'{vm["pearson"]:.3f}',
                'best':    f'ep{self.best_epoch}',
                '★' if is_best else '': ''
            })

            # ── Auto-save every N epochs in case Colab crashes ───────────
            if epoch % self.cfg.save_interval == 0:
                self.save_model(f'epoch_{epoch}.pt')
                self.save_history()

        self.save_history()
        print(f'\n✅ Training complete!')
        print(f'   Best epoch : {self.best_epoch}/{total}')
        print(f'   Best val MSE : {self.best_mse:.4f}')
        print(f'   Best val RMSE: {np.sqrt(self.best_mse):.4f}')

print("Trainer defined.")


Trainer defined.


## ✅ CELL 12 — Load Data, Split, Build Loaders

## ✅ CELL 11b — Fetch pIC50 from ChEMBL API (run once)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# FETCH pIC50 FROM ChEMBL API  (run once — saves merged CSV to Drive)
# ═══════════════════════════════════════════════════════════════════════════
import requests, time, pandas as pd, numpy as np
from pathlib import Path

SELFIES_CSV  = f'{DRIVE_ROOT}/smiles_to_selfies_output.csv'
MERGED_CSV   = f'{DRIVE_ROOT}/lassapic_with_pic50.csv'

if Path(MERGED_CSV).exists():
    print('✅ Merged CSV already on Drive — skipping fetch.')
else:
    df_drug = pd.read_csv(SELFIES_CSV)
    print(f'Drug file columns : {list(df_drug.columns)}')
    print(f'Rows              : {len(df_drug)}')

    # ── Identify the ChEMBL ID column ──────────────────────────────────────
    id_col = None
    for c in ['Molecule ChEMBL ID','chembl_id','ChEMBL ID','molecule_chembl_id']:
        if c in df_drug.columns:
            id_col = c
            break
    if id_col is None:
        raise ValueError(f"No ChEMBL ID column found. Columns: {list(df_drug.columns)}")
    print(f'Using ID column   : {id_col}')

    chembl_ids = df_drug[id_col].dropna().unique().tolist()
    print(f'Unique ChEMBL IDs : {len(chembl_ids)}')

    # ── Fetch bioactivities in batches of 50 ───────────────────────────────
    BASE = "https://www.ebi.ac.uk/chembl/api/data/activity.json"
    records = []
    BATCH   = 50

    for start in range(0, len(chembl_ids), BATCH):
        batch = chembl_ids[start:start+BATCH]
        ids_str = ';'.join(batch)
        params  = {
            'molecule_chembl_id__in': ids_str,
            'standard_type__in':      'IC50;Ki;Kd;EC50',
            'standard_relation__in':  '=;<',
            'standard_units':         'nM',
            'limit':                  1000,
            'offset':                 0,
        }
        try:
            r = requests.get(BASE, params=params, timeout=30)
            r.raise_for_status()
            data = r.json()
            records.extend(data['activities'])
            print(f'  Batch {start//BATCH+1}: fetched {len(data["activities"])} activities')
        except Exception as e:
            print(f'  Batch {start//BATCH+1}: ERROR — {e}')
        time.sleep(0.3)   # be kind to the API

    if not records:
        print('\n⚠️  No activities returned from ChEMBL.')
        print('This can happen if your molecules have no IC50/Ki data in ChEMBL,')
        print('or if the target assay uses different units (e.g. % inhibition).')
        print('Falling back to synthetic pIC50 for pipeline testing.')
        np.random.seed(42)
        df_drug['pIC50'] = np.random.uniform(4.5, 9.5, len(df_drug)).round(2)
        df_drug.to_csv(MERGED_CSV, index=False)
        print(f'Saved {len(df_drug)} rows with synthetic pIC50 → {MERGED_CSV}')
    else:
        act_df = pd.DataFrame(records)
        print(f'\nTotal activity records fetched: {len(act_df)}')
        print(f'Columns: {list(act_df.columns)}')

        # Keep only valid numeric values
        act_df['standard_value'] = pd.to_numeric(act_df['standard_value'], errors='coerce')
        act_df = act_df.dropna(subset=['standard_value'])
        act_df = act_df[act_df['standard_value'] > 0]

        # Convert to pIC50 (values are in nM)
        act_df['pIC50'] = -np.log10(act_df['standard_value'].clip(lower=0.001) * 1e-9)

        # Take median pIC50 per molecule (some have multiple assays)
        agg = (act_df
               .groupby('molecule_chembl_id')['pIC50']
               .median()
               .reset_index()
               .rename(columns={'molecule_chembl_id': id_col}))
        print(f'Unique molecules with activity: {len(agg)}')

        merged = df_drug.merge(agg, on=id_col, how='inner')
        print(f'Merged rows (drug ∩ activity): {len(merged)}')
        print(f'pIC50 range: {merged["pIC50"].min():.2f} – {merged["pIC50"].max():.2f}')

        merged.to_csv(MERGED_CSV, index=False)
        print(f'\n✅ Saved merged CSV → {MERGED_CSV}')

print('Done.')


✅ Merged CSV already on Drive — skipping fetch.
Done.


In [ ]:
# ── DIAGNOSTIC: inspect your CSV columns before loading ───────────────────
import pandas as pd
_tmp = pd.read_csv(config.data_path, nrows=3)
print("Columns found:", list(_tmp.columns))
print("\nFirst 3 rows:")
print(_tmp.to_string())
print("\n⬆ Copy the exact column name that contains your IC50 / pIC50 / activity values")
print("   then set  ACTIVITY_COL  below in Cell 12.")


Columns found: ['Molecule ChEMBL ID', 'Smiles', 'SELFIES', 'pIC50']

First 3 rows:
  Molecule ChEMBL ID                                                                                                               Smiles                                                                                                                                                                                                                                                                                                                                                                                                                                   SELFIES  pIC50
0        CHEMBL78781  CCC[C@H](NC(=O)OC(C)(C)C)C(=O)N[C@@H](C)C(=O)N[C@@H](Cc1ccccc1)[C@@H](O)C[C@@H](C)C(=O)N[C@H](C(=O)NCc1ccncc1)C(C)C                           [C][C][C][C@H1][Branch1][#C][N][C][=Branch1][C][=O][O][C][Branch1][C][C][Branch1][C][C][C][C][=Branch1][C][=O][N][C@@H1][Branch1][C][C][C][=Branch1][C][=O][N][C@@H1][Branch1][#Branch

In [ ]:
# ── FIX ⑮  Force reprocess when proc_data was built with the old buggy code ──
# Delete the cached file once, then comment this out.
if PROC_DATA.exists():
    PROC_DATA.unlink()
    logger.info("Deleted stale processed_data.csv — will reprocess with fixes.")

# ▼▼▼  SET THIS to the exact column name shown by the diagnostic cell above  ▼▼▼
# Common names: 'IC50', 'pIC50', 'Ki', 'Kd', 'activity', 'standard_value', 'Value'
# Leave as None to let the code auto-detect from the list above.
ACTIVITY_COL = None   # e.g.  ACTIVITY_COL = "standard_value"

# ── Auto-detect if not set ────────────────────────────────────────────────
_KNOWN = ['pIC50','pic50','IC50','ic50','Ki','ki','Kd','kd',
          'activity','Activity','standard_value','Value','value',
          'affinity','Affinity','score','Score']

if ACTIVITY_COL is None:
    _peek = pd.read_csv(config.data_path, nrows=0)
    for _c in _KNOWN:
        if _c in _peek.columns:
            ACTIVITY_COL = _c
            break
    if ACTIVITY_COL is None:
        raise ValueError(
            f"Cannot auto-detect activity column.\n"
            f"Columns found: {list(_peek.columns)}\n"
            f"Set ACTIVITY_COL manually at the top of this cell."
        )

logger.info(f"Using activity column: '{ACTIVITY_COL}'")

processor = DataProcessor(config, activity_col=ACTIVITY_COL)

if PROC_DATA.exists():
    logger.info('✅ Loading cached processed_data.csv from Drive')
    df = pd.read_csv(PROC_DATA)
else:
    logger.info('Processing raw data...')
    df = processor.load_and_process(config.data_path)
    df.to_csv(PROC_DATA, index=False)
    logger.info(f'Saved processed_data.csv → {PROC_DATA}')

logger.info(f'Dataset: {df.shape} | pIC50 range: {df["pIC50"].min():.2f}–{df["pIC50"].max():.2f}')

tv_df, test_df  = train_test_split(df,    test_size=config.test_split, random_state=42)
val_sz           = config.val_split / (1 - config.test_split)
train_df, val_df = train_test_split(tv_df, test_size=val_sz,           random_state=42)
logger.info(f'Train={len(train_df)} | Val={len(val_df)} | Test={len(test_df)}')

# FIX ⑯  Z-score normalise pIC50 from training data only
pic50_mean = float(train_df['pIC50'].mean())
pic50_std  = float(train_df['pIC50'].std() + 1e-8)
config.pic50_mean = pic50_mean
config.pic50_std  = pic50_std
logger.info(f'pIC50 → mean={pic50_mean:.3f}  std={pic50_std:.3f}')

for split in [train_df, val_df, test_df]:
    split['pIC50_norm'] = (split['pIC50'] - pic50_mean) / pic50_std

train_loader = make_loader(DTADataset(train_df, config), config.batch_size, shuffle=True)
val_loader   = make_loader(DTADataset(val_df,   config), config.batch_size, shuffle=False)
test_loader  = make_loader(DTADataset(test_df,  config), config.batch_size, shuffle=False)
print('Loaders ready.')


Concepts: 100%|██████████| 5092/5092 [00:35<00:00, 143.38it/s]


Loaders ready.


## ✅ CELL 13 — Build Model

In [ ]:
model = DTAModel(config).to(device)
print(f'Trainable parameters: {model.n_params():,}')

config.json:   0%|          | 0.00/634 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/31.4M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/107 [00:00<?, ?it/s]

EsmModel LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                         | Status     | 
----------------------------+------------+-
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
pooler.dense.bias           | MISSING    | 
pooler.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Trainable parameters: 94,871,814


## ✅ CELL 14 — Train ONCE (skips if best_model.pt already on Drive)

## 🔴 FIX: Replace Synthetic pIC50 with Real Bioactivity Data

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# STEP 1: LOAD REAL DAVIS DTA DATA  (direct download, no extra libraries)
# ══════════════════════════════════════════════════════════════════════════
import requests, json as _json
import pandas as pd
import numpy as np
import selfies as sf
from pathlib import Path

REAL_DATA_CSV = Path(config.output_dir) / 'real_dta_data.csv'

if REAL_DATA_CSV.exists():
    print(f"✅ Real dataset already cached → {REAL_DATA_CSV}")
    df_real = pd.read_csv(REAL_DATA_CSV)
else:
    # ── GraphDTA repo — Davis dataset (stable, well-known benchmark paper) ──
    BASE = "https://raw.githubusercontent.com/thinng/GraphDTA/master/data/davis"
    print("Downloading Davis dataset from GraphDTA repo...")

    def fetch(url):
        r = requests.get(url, timeout=30)
        r.raise_for_status()
        return r.text.strip()

    try:
        # Drug SMILES (68 unique drugs)
        smiles_raw  = fetch(f"{BASE}/ligands_can.txt")
        # Protein sequences (442 unique targets)
        protein_raw = fetch(f"{BASE}/proteins.txt")
        # Affinity matrix 442×68  (Kd values in nM, -1 = missing)
        affinity_raw = fetch(f"{BASE}/Y")

        # Parse
        import ast
        smiles_dict  = ast.literal_eval(smiles_raw)   # dict {name: smiles}
        protein_dict = ast.literal_eval(protein_raw)  # dict {name: seq}
        aff_lines    = [line.split() for line in affinity_raw.splitlines()]

        drugs    = list(smiles_dict.values())
        proteins = list(protein_dict.values())
        aff_mat  = np.array([[float(x) for x in row] for row in aff_lines])

        print(f"  Drugs    : {len(drugs)}")
        print(f"  Proteins : {len(proteins)}")
        print(f"  Affinity : {aff_mat.shape}  (should be 442×68)")

        rows = []
        for p_i, prot_seq in enumerate(proteins):
            for d_i, drug_smi in enumerate(drugs):
                kd = aff_mat[p_i, d_i]
                if kd > 0:
                    pkd = -np.log10(kd * 1e-9 + 1e-10)
                    rows.append({"Smiles": drug_smi,
                                 "sequence": prot_seq,
                                 "pIC50": round(pkd, 4),
                                 "Kd_nM": kd})

        df_real = pd.DataFrame(rows)
        df_real = df_real[df_real["pIC50"].between(2, 12)].copy()
        print(f"  Valid pairs: {len(df_real)}")

    except Exception as e1:
        print(f"GraphDTA failed ({e1}), trying BindingDB ChEMBL API...")
        # Fallback: pull real IC50s from ChEMBL for EGFR (well-studied target)
        TARGET = "CHEMBL203"   # EGFR — thousands of IC50 measurements
        url = (f"https://www.ebi.ac.uk/chembl/api/data/activity.json"
               f"?target_chembl_id={TARGET}"
               f"&standard_type=IC50&standard_units=nM"
               f"&standard_relation==&limit=500")
        r = requests.get(url, timeout=30); r.raise_for_status()
        acts = r.json()["activities"]
        rows = []
        for a in acts:
            try:
                val = float(a["standard_value"])
                smi = a.get("canonical_smiles") or a.get("molecule_smiles","")
                if val > 0 and smi:
                    pkd = -np.log10(val * 1e-9)
                    rows.append({"Smiles": smi,
                                 "sequence": "MRPSGTAGAALLALLAALCPASRALEEKKVC"
                                             "QGTSNKLTQLGTFEDHFLSLQRMFNNCEVVL"
                                             "GNLEITYVQRNYDLSFLKTIQEVAGYVLIAL"
                                             "NTVERIPLENLQIIRGNMYYENSYALAVLSN"
                                             "YDANKTGLKELPMRNLQEILHGAVRFSNNPA"
                                             "LLFSTMPSSTVK",   # EGFR sequence
                                 "pIC50": round(pkd, 4)})
            except: pass
        df_real = pd.DataFrame(rows)
        df_real = df_real[df_real["pIC50"].between(2, 12)].copy()
        print(f"  ChEMBL EGFR pairs: {len(df_real)}")

    # ── SELFIES ────────────────────────────────────────────────────────────
    print("Converting SMILES → SELFIES...")
    sf_list = []
    for smi in df_real["Smiles"]:
        try:    sf_list.append(sf.encoder(str(smi)))
        except: sf_list.append(None)
    df_real["selfies"] = sf_list
    df_real = df_real[df_real["selfies"].notna()].copy()

    df_real.to_csv(REAL_DATA_CSV, index=False)
    print(f"✅ Saved {len(df_real)} rows → {REAL_DATA_CSV}")

# ── Update pipeline ────────────────────────────────────────────────────────
config.data_path = str(REAL_DATA_CSV)
df = df_real.copy()

print(f'\n{"="*50}')
print(f'Rows   : {len(df)}')
print(f'Mean   : {df["pIC50"].mean():.3f}')
print(f'Std    : {df["pIC50"].std():.3f}')
print(f'Range  : {df["pIC50"].min():.3f} – {df["pIC50"].max():.3f}')

import matplotlib.pyplot as plt
plt.figure(figsize=(7,3))
plt.hist(df['pIC50'], bins=40, color='steelblue', edgecolor='white')
plt.axvline(df['pIC50'].mean(), color='red', ls='--',
            label=f'mean={df["pIC50"].mean():.2f}')
plt.xlabel('pIC50'); plt.ylabel('Count'); plt.legend()
plt.title('pIC50 distribution — bell-shaped = real data ✅')
plt.tight_layout(); plt.show()
print("\n✅ Run Step 2 next (Reprocess cell).")


✅ Real dataset already cached → /content/drive/MyDrive/DTA_CBM/outputs/real_dta_data.csv

Rows   : 500
Mean   : 6.483
Std    : 1.713
Range  : 2.187 – 11.222

✅ Run Step 2 next (Reprocess cell).


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# STEP 2: RECOMPUTE CONCEPTS + SPLITS FOR REAL DATA
# (delete stale embeddings so they are recomputed on real data)
# ══════════════════════════════════════════════════════════════════════════
from pathlib import Path
import numpy as np

EMBED_DIR  = Path(config.output_dir) / 'embeddings'
DRUG_EMB_F = EMBED_DIR / 'drug_embeddings.npy'
PROT_EMB_F = EMBED_DIR / 'prot_embeddings.npy'
IDX_F      = EMBED_DIR / 'row_index.npy'

# Delete stale embeddings built on synthetic data
for f in [DRUG_EMB_F, PROT_EMB_F, IDX_F]:
    if f.exists():
        f.unlink()
        print(f"Deleted stale embedding: {f.name}")

# Delete stale processed_data and best_model (synthetic labels)
for f in [PROC_DATA, BEST_MODEL, HIST_FILE,
          Path(config.model_save_dir) / 'best_model_fast.pt']:
    if f.exists():
        f.unlink()
        print(f"Deleted stale file: {f.name}")

# Compute drug concepts on real SMILES
processor = DataProcessor(config, activity_col='pIC50')
print("\nComputing drug concepts on real dataset...")
cdata = []
for _, row in df.iterrows():
    d = processor.calc_drug_concepts(row['Smiles'])
    seq = row['sequence'] if 'sequence' in row.index else _DEFAULT_PROTEIN_SEQ
    p = processor.calc_protein_concepts(seq)
    merged = {**d, **{k: v for k, v in p.items() if k in ['zinc_binding','steric_clash']}}
    cdata.append(merged)
cdf = pd.DataFrame(cdata)
for col in cdf.columns:
    df[col] = cdf[col].values

# Splits
from sklearn.model_selection import train_test_split
tv_df, test_df  = train_test_split(df,    test_size=config.test_split, random_state=42)
val_sz           = config.val_split / (1 - config.test_split)
train_df, val_df = train_test_split(tv_df, test_size=val_sz,           random_state=42)

# pIC50 normalisation from training set
pic50_mean = float(train_df['pIC50'].mean())
pic50_std  = float(train_df['pIC50'].std() + 1e-8)
config.pic50_mean = pic50_mean
config.pic50_std  = pic50_std
for split in [train_df, val_df, test_df]:
    split['pIC50_norm'] = (split['pIC50'] - pic50_mean) / pic50_std

print(f"\nTrain={len(train_df)} | Val={len(val_df)} | Test={len(test_df)}")
print(f"pIC50 mean={pic50_mean:.3f}  std={pic50_std:.3f}")
print("\n✅ Done. Now run the Pre-compute Embeddings cell (cell 31).")


Deleted stale embedding: drug_embeddings.npy
Deleted stale embedding: prot_embeddings.npy
Deleted stale embedding: row_index.npy
Deleted stale file: processed_data.csv
Deleted stale file: training_history.json
Deleted stale file: best_model_fast.pt

Computing drug concepts on real dataset...

Train=350 | Val=50 | Test=100
pIC50 mean=6.477  std=1.736

✅ Done. Now run the Pre-compute Embeddings cell (cell 31).


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# PRE-COMPUTE EMBEDDINGS  (run once — saves ~10 min, saves 50× per epoch)
# Drug embeddings  : SELFormer over all SELFIES strings
# Protein embeddings: ESM-2    over all (unique) sequences
# After this cell every training batch takes <0.5s instead of 30s
# ══════════════════════════════════════════════════════════════════════
import numpy as np, torch, os
from pathlib import Path

EMBED_DIR  = Path(config.output_dir) / 'embeddings'
EMBED_DIR.mkdir(parents=True, exist_ok=True)

DRUG_EMB_F = EMBED_DIR / 'drug_embeddings.npy'
PROT_EMB_F = EMBED_DIR / 'prot_embeddings.npy'
IDX_F      = EMBED_DIR / 'row_index.npy'       # maps df row → embedding row

model.eval()

# ── Helper: mean-pool last hidden state ──────────────────────────────────
def mean_pool(hidden, attention_mask):
    mask = attention_mask.unsqueeze(-1).float()
    return (hidden * mask).sum(1) / mask.sum(1).clamp(min=1e-9)

# ── 1. Drug embeddings ───────────────────────────────────────────────────
if DRUG_EMB_F.exists() and IDX_F.exists():
    print('✅ Drug embeddings already cached — skipping.')
    drug_embs = np.load(DRUG_EMB_F)
else:
    print('Computing drug embeddings (SELFormer)...')
    all_selfies = df['selfies'].tolist()
    drug_embs   = np.zeros((len(df), config.drug_dim), dtype=np.float32)
    BS = 32
    with torch.no_grad():
        for start in tqdm.tqdm(range(0, len(all_selfies), BS), desc='Drug emb'):
            batch_sf = all_selfies[start:start+BS]
            enc = model.drug_enc.tokenizer(
                batch_sf,
                return_tensors='pt',
                padding=True,
                truncation=True,
                max_length=config.max_drug_length
            ).to(device)
            out = (model.drug_enc.encoder(**enc)
                   if hasattr(model.drug_enc, 'encoder')
                   else model.drug_enc.model(**enc))
            emb = mean_pool(out.last_hidden_state, enc['attention_mask'])
            drug_embs[start:start+BS] = emb.cpu().numpy()
    np.save(DRUG_EMB_F, drug_embs)
    np.save(IDX_F, np.arange(len(df)))
    print(f'   Saved drug embeddings {drug_embs.shape} → {DRUG_EMB_F}')

# ── 2. Protein embeddings (unique seqs only) ─────────────────────────────
seq_col = 'sequence' if 'sequence' in df.columns else 'protein_sequence'
if PROT_EMB_F.exists():
    print('✅ Protein embeddings already cached — skipping.')
    prot_embs_full = np.load(PROT_EMB_F)
else:
    print('Computing protein embeddings (ESM-2)...')
    seqs       = df[seq_col].tolist()
    unique_seqs = list(dict.fromkeys(seqs))   # preserve order, deduplicate
    seq2idx     = {s: i for i, s in enumerate(unique_seqs)}
    uniq_embs   = np.zeros((len(unique_seqs), config.protein_dim), dtype=np.float32)
    BS = 8
    with torch.no_grad():
        for start in tqdm.tqdm(range(0, len(unique_seqs), BS), desc='Prot emb'):
            batch_seq = unique_seqs[start:start+BS]
            enc = model.prot_enc.tokenizer(
                batch_seq,
                return_tensors='pt',
                padding=True,
                truncation=True,
                max_length=config.max_protein_length
            ).to(device)
            out = (model.prot_enc.encoder(**enc)
                   if hasattr(model.prot_enc, 'encoder')
                   else model.prot_enc.model(**enc))
            emb = mean_pool(out.last_hidden_state, enc['attention_mask'])
            uniq_embs[start:start+BS] = emb.cpu().numpy()
    # expand back to full df length
    prot_embs_full = np.array([uniq_embs[seq2idx[s]] for s in seqs], dtype=np.float32)
    np.save(PROT_EMB_F, prot_embs_full)
    print(f'   Saved protein embeddings {prot_embs_full.shape} → {PROT_EMB_F}')

print(f'\nDrug emb shape  : {drug_embs.shape}')
print(f'Prot emb shape  : {prot_embs_full.shape}')
print('✅ Embeddings ready. Training will now be fast.')


Computing drug embeddings (SELFormer)...


Drug emb: 100%|██████████| 16/16 [03:00<00:00, 11.27s/it]


   Saved drug embeddings (500, 768) → /content/drive/MyDrive/DTA_CBM/outputs/embeddings/drug_embeddings.npy
Computing protein embeddings (ESM-2)...


Prot emb: 100%|██████████| 1/1 [00:00<00:00, 11.75it/s]

   Saved protein embeddings (500, 320) → /content/drive/MyDrive/DTA_CBM/outputs/embeddings/prot_embeddings.npy

Drug emb shape  : (500, 768)
Prot emb shape  : (500, 320)
✅ Embeddings ready. Training will now be fast.


In [ ]:
# Fast Dataset — loads pre-computed embeddings instead of running encoders
class FastDTADataset(Dataset):
    """Uses cached numpy embeddings — each __getitem__ is a simple array lookup."""
    def __init__(self, df_split, drug_embs, prot_embs, config):
        self.df        = df_split.reset_index(drop=True)
        self.drug_embs = drug_embs   # shape (N_total, drug_dim)
        self.prot_embs = prot_embs   # shape (N_total, prot_dim)
        self.cfg       = config
        # Map split row → original df index (used to look up embeddings)
        self.orig_idx  = df_split.index.tolist()

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        orig = self.orig_idx[i]
        r    = self.df.iloc[i]
        c    = np.array([float(r[n]) if n in r.index else 0.0
                         for n in self.cfg.concept_names], dtype=np.float32)
        return {
            'drug_emb': torch.FloatTensor(self.drug_embs[orig]),
            'prot_emb': torch.FloatTensor(self.prot_embs[orig]),
            'concepts': torch.FloatTensor(c),
            'pic50':    torch.FloatTensor([float(r['pIC50_norm'])]),
        }

def fast_collate(batch):
    return {
        'drug_emb': torch.stack([b['drug_emb'] for b in batch]),
        'prot_emb': torch.stack([b['prot_emb'] for b in batch]),
        'concepts': torch.stack([b['concepts'] for b in batch]),
        'pic50':    torch.stack([b['pic50']    for b in batch]),
    }

def make_fast_loader(df_split, drug_embs, prot_embs, bs, shuffle):
    ds = FastDTADataset(df_split, drug_embs, prot_embs, config)
    return DataLoader(ds, batch_size=bs, shuffle=shuffle,
                      collate_fn=fast_collate, num_workers=0, pin_memory=True)

# Re-create data splits using original df indices
tv_df2, test_df2  = train_test_split(df, test_size=config.test_split, random_state=42)
val_sz2            = config.val_split / (1 - config.test_split)
train_df2, val_df2 = train_test_split(tv_df2, test_size=val_sz2, random_state=42)

# Normalise pIC50
pic50_mean = float(train_df2['pIC50'].mean())
pic50_std  = float(train_df2['pIC50'].std() + 1e-8)
config.pic50_mean = pic50_mean
config.pic50_std  = pic50_std
for split in [train_df2, val_df2, test_df2]:
    split['pIC50_norm'] = (split['pIC50'] - pic50_mean) / pic50_std

train_loader = make_fast_loader(train_df2, drug_embs, prot_embs_full, config.batch_size, shuffle=True)
val_loader   = make_fast_loader(val_df2,   drug_embs, prot_embs_full, config.batch_size, shuffle=False)
test_loader  = make_fast_loader(test_df2,  drug_embs, prot_embs_full, config.batch_size, shuffle=False)

print(f'Train={len(train_df2)} | Val={len(val_df2)} | Test={len(test_df2)}')
print(f'pIC50 mean={pic50_mean:.3f}  std={pic50_std:.3f}')
print('✅ Fast loaders ready — each batch is now <0.5s')


Train=350 | Val=50 | Test=100
pIC50 mean=6.477  std=1.736
✅ Fast loaders ready — each batch is now <0.5s


In [ ]:
# Fast CBM + predictor head (no encoder — embeddings come in pre-computed)
class FastCBM(nn.Module):
    def __init__(self, config):
        super().__init__()
        fd = config.fused_dim   # 1088

        self.net = nn.Sequential(
            nn.Linear(fd, 512),
            nn.BatchNorm1d(512),
            nn.GELU(),
            nn.Dropout(config.dropout_rate),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.GELU(),
            nn.Dropout(config.dropout_rate),
            nn.Linear(256, config.num_concepts),
            nn.Sigmoid(),
        )

    def forward(self, fused):
        return self.net(fused)

class FastPredictor(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.predictor = nn.Sequential(
            nn.Linear(config.num_concepts, 64),
            nn.GELU(),
            nn.Dropout(config.dropout_rate),
            nn.Linear(64, 32),
            nn.GELU(),
            nn.Dropout(config.dropout_rate / 2),
            nn.Linear(32, 1),
        )

    def forward(self, concepts):
        return self.predictor(concepts)

class FastDTAModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.cbm  = FastCBM(config)
        self.pred = FastPredictor(config)

    def forward(self, drug_emb, prot_emb):
        fused    = torch.cat([drug_emb, prot_emb], dim=-1)   # (B, 1088)
        concepts = self.cbm(fused)
        affinity = self.pred(concepts)
        return affinity, concepts

fast_model = FastDTAModel(config).to(device)
print(f'FastDTAModel parameters: {sum(p.numel() for p in fast_model.parameters()):,}')

# ── Fast Trainer ──────────────────────────────────────────────────────────
class FastTrainer:
    def __init__(self, model, config):
        self.model = model
        self.cfg   = config
        self.loss  = CBMLoss(config)
        self.opt   = AdamW(model.parameters(), lr=config.head_lr,
                           weight_decay=config.weight_decay)
        self.sch   = CosineAnnealingLR(self.opt, T_max=config.num_epochs, eta_min=1e-6)
        self.history   = {k: [] for k in
                          ['train_loss','val_loss','val_mse','val_r2','val_pearson','val_spearman']}
        self.best_mse   = float('inf')
        self.best_epoch = -1

    def save(self, name):
        torch.save({'model_state_dict': self.model.state_dict(),
                    'best_mse':  self.best_mse,
                    'best_epoch': self.best_epoch,
                    'pic50_mean': self.cfg.pic50_mean,
                    'pic50_std':  self.cfg.pic50_std},
                   Path(self.cfg.model_save_dir) / name)

    def save_history(self):
        # Guard: ensure each history entry is a list, not a stale scalar
        payload = {k: [float(v) for v in (vs if isinstance(vs, list) else [vs])]
                   for k, vs in self.history.items()}
        payload['pic50_mean'] = float(self.cfg.pic50_mean)
        payload['pic50_std']  = float(self.cfg.pic50_std)
        with open(HIST_FILE, 'w') as f: json.dump(payload, f, indent=2)

    def _step(self, batch, train=True):
        de = batch['drug_emb'].to(device)
        pe = batch['prot_emb'].to(device)
        tc = batch['concepts'].to(device)
        ty = batch['pic50'].to(device)
        py, pc = self.model(de, pe)
        losses = self.loss(pc, tc, py, ty)
        if train:
            self.opt.zero_grad()
            losses['total'].backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
            self.opt.step()
        return losses['total'].item(), py.detach(), ty.detach(), pc.detach(), tc.detach()

    def run_epoch(self, loader, train, tag):
        self.model.train() if train else self.model.eval()
        ctx = torch.enable_grad() if train else torch.no_grad()
        total, pn, tn, cp_list, ct_list = 0.0, [], [], [], []
        with ctx:
            for b in loader:
                l, py, ty, pc, tc = self._step(b, train)
                total += l
                pn.extend(py.squeeze().cpu().tolist() if py.squeeze().ndim > 0 else [py.squeeze().item()])
                tn.extend(ty.squeeze().cpu().tolist() if ty.squeeze().ndim > 0 else [ty.squeeze().item()])
                cp_list.append(pc.cpu().numpy())
                ct_list.append(tc.cpu().numpy())

        mu, sigma = self.cfg.pic50_mean, self.cfg.pic50_std
        preds = np.array(pn) * sigma + mu
        trues = np.array(tn) * sigma + mu
        mse   = mean_squared_error(trues, preds)
        r2    = r2_score(trues, preds)
        pr, _ = pearsonr(trues, preds)
        sp, _ = spearmanr(trues, preds)
        return {'loss': total/len(loader), 'mse': mse, 'r2': r2,
                'pearson': pr, 'spearman': sp,
                'preds': preds, 'trues': trues,
                'cp': np.concatenate(cp_list), 'ct': np.concatenate(ct_list)}

    def train(self, train_loader, val_loader):
        total = self.cfg.num_epochs
        print(f'\n🚀 Training {total} epochs | {len(train_loader)} batches/epoch')
        print(f'   Estimated time: ~{len(train_loader)*total*0.05/60:.0f} min on GPU\n')

        epoch_bar = tqdm.tqdm(range(1, total+1), desc='Overall',
                              unit='epoch', leave=True,
                              bar_format='{l_bar}{bar:30}{r_bar}')
        for epoch in epoch_bar:
            tm = self.run_epoch(train_loader, train=True,  tag='train')
            vm = self.run_epoch(val_loader,   train=False, tag='val')
            self.sch.step()
            is_best = vm['mse'] < self.best_mse
            if is_best:
                self.best_mse = vm['mse']; self.best_epoch = epoch
                self.save('best_model_fast.pt')
            self.history['train_loss'].append(tm['loss'])
            self.history['val_loss'].append(vm['loss'])
            self.history['val_mse'].append(vm['mse'])
            self.history['val_r2'].append(vm['r2'])
            self.history['val_pearson'].append(vm['pearson'])
            self.history['val_spearman'].append(vm['spearman'])
            epoch_bar.set_postfix({
                't_loss': f'{tm["loss"]:.3f}',
                'v_mse':  f'{vm["mse"]:.3f}',
                'R²':     f'{vm["r2"]:.3f}',
                'r':      f'{vm["pearson"]:.3f}',
                'best':   f'ep{self.best_epoch}' + (' ★' if is_best else ''),
            })
            if epoch % self.cfg.save_interval == 0:
                self.save(f'fast_epoch_{epoch}.pt')
                self.save_history()
        self.save_history()
        print(f'\n✅ Done! Best epoch={self.best_epoch} | MSE={self.best_mse:.4f} | RMSE={np.sqrt(self.best_mse):.4f}')

    def evaluate(self, loader, split='test'):
        res = self.run_epoch(loader, train=False, tag=split)
        cp, ct = res['cp'], res['ct']
        concept_acc = {}
        for idx in self.cfg.binary_concepts:
            pred_b = (cp[:, idx] > 0.5).astype(float)
            concept_acc[self.cfg.concept_names[idx]] = float(np.mean(pred_b == ct[:, idx]))
        res['concept_accuracy'] = concept_acc
        return res

FAST_MODEL = Path(config.model_save_dir) / 'best_model_fast.pt'
fast_trainer = FastTrainer(fast_model, config)

if FAST_MODEL.exists():
    ckpt = torch.load(FAST_MODEL, map_location=device)
    fast_model.load_state_dict(ckpt['model_state_dict'])
    fast_trainer.best_mse   = ckpt.get('best_mse',   float('inf'))
    fast_trainer.best_epoch = ckpt.get('best_epoch',  -1)
    if 'pic50_mean' in ckpt:
        config.pic50_mean = ckpt['pic50_mean']
        config.pic50_std  = ckpt['pic50_std']
    print(f'✅ Loaded fast checkpoint (epoch {fast_trainer.best_epoch}, MSE {fast_trainer.best_mse:.4f})')
else:
    fast_trainer.train(train_loader, val_loader)


FastDTAModel parameters: 696,461

🚀 Training 50 epochs | 22 batches/epoch
   Estimated time: ~1 min on GPU



Overall: 100%|██████████████████████████████| 50/50 [00:21<00:00,  2.33epoch/s, t_loss=0.224, v_mse=0.884, R²=0.667, r=0.818, best=ep39]


✅ Done! Best epoch=39 | MSE=0.8692 | RMSE=0.9323


In [ ]:
print('Evaluating on test set...')
tm = fast_trainer.evaluate(test_loader, split='test')

print('\n' + '='*58)
print('             TEST SET METRICS')
print('='*58)
print(f'  MSE               : {tm["mse"]:.4f}')
print(f'  RMSE              : {np.sqrt(tm["mse"]):.4f}')
print(f'  R² Score          : {tm["r2"]:.4f}')
print(f'  Pearson r         : {tm["pearson"]:.4f}')
print(f'  Spearman ρ        : {tm["spearman"]:.4f}')
print('\n  --- Concept Accuracy (Binary Concepts) ---')
mean_acc = np.mean(list(tm['concept_accuracy'].values()))
for name, acc in tm['concept_accuracy'].items():
    bar = '█' * int(acc * 24)
    print(f'  {name:<30} {acc*100:5.1f}%  {bar}')
print(f'  {"─"*50}')
print(f'  Mean Concept Accuracy         {mean_acc*100:.1f}%')


Evaluating on test set...

             TEST SET METRICS
  MSE               : 0.8114
  RMSE              : 0.9008
  R² Score          : 0.7091
  Pearson r         : 0.8472
  Spearman ρ        : 0.8182

  --- Concept Accuracy (Binary Concepts) ---
  zinc_binding                    86.0%  ████████████████████
  hydrogen_bond_donor             95.0%  ██████████████████████
  hydrogen_bond_acceptor          93.0%  ██████████████████████
  hydrophobic_contact             97.0%  ███████████████████████
  steric_clash                   100.0%  ████████████████████████
  ──────────────────────────────────────────────────
  Mean Concept Accuracy         94.2%


In [ ]:
# ── OLD SLOW TRAIN CELL — DISABLED (replaced by Fast Trainer at cells 31-34) ──
# The old trainer ran transformers on every batch (30s/batch).
# The fast trainer below uses cached embeddings (~0.05s/batch).
# Do not run this cell.
print("Old slow train cell — skipped. Use the Fast Trainer cells above.")


Old slow train cell — skipped. Use the Fast Trainer cells above.


## ✅ CELL 15 — Test Set Evaluation + Full Metrics

In [ ]:
# ── TEST EVALUATION — uses fast_trainer ───────────────────────────────────
print("Evaluating on test set...")
tm = fast_trainer.evaluate(test_loader, split='test')

print('\n' + '='*58)
print('             TEST SET METRICS')
print('='*58)
print(f'  MSE               : {tm["mse"]:.4f}')
print(f'  RMSE              : {np.sqrt(tm["mse"]):.4f}')
print(f'  R² Score          : {tm["r2"]:.4f}')
print(f'  Pearson r         : {tm["pearson"]:.4f}')
print(f'  Spearman ρ        : {tm["spearman"]:.4f}')
print('\n  --- Concept Accuracy (Binary Concepts) ---')
mean_acc = np.mean(list(tm['concept_accuracy'].values()))
for name, acc in tm['concept_accuracy'].items():
    bar = '█' * int(acc * 24)
    print(f'  {name:<30} {acc*100:5.1f}%  {bar}')
print(f'  {"─"*50}')
print(f'  Mean Concept Accuracy         {mean_acc*100:.1f}%')


Evaluating on test set...

             TEST SET METRICS
  MSE               : 0.8114
  RMSE              : 0.9008
  R² Score          : 0.7091
  Pearson r         : 0.8472
  Spearman ρ        : 0.8182

  --- Concept Accuracy (Binary Concepts) ---
  zinc_binding                    86.0%  ████████████████████
  hydrogen_bond_donor             95.0%  ██████████████████████
  hydrogen_bond_acceptor          93.0%  ██████████████████████
  hydrophobic_contact             97.0%  ███████████████████████
  steric_clash                   100.0%  ████████████████████████
  ──────────────────────────────────────────────────
  Mean Concept Accuracy         94.2%


## ✅ CELL 16 — Graphs (never empty — loaded from Drive history)

In [ ]:
# ── TRAINING CURVES ────────────────────────────────────────────────────────
h = fast_trainer.history
epochs = list(range(1, len(h['train_loss']) + 1))
if not epochs:
    print("No history yet — train first.")
else:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    axes[0].plot(epochs, h['train_loss'], label='Train Loss')
    axes[0].plot(epochs, h['val_loss'],   label='Val Loss')
    axes[0].set_title('Loss'); axes[0].legend(); axes[0].set_xlabel('Epoch')

    axes[1].plot(epochs, h['val_mse'], color='orange', label='Val MSE')
    axes[1].set_title('Validation MSE'); axes[1].legend(); axes[1].set_xlabel('Epoch')

    axes[2].plot(epochs, h['val_r2'],      label='R²')
    axes[2].plot(epochs, h['val_pearson'], label='Pearson r')
    axes[2].axhline(0, color='gray', lw=0.8, ls='--')
    axes[2].set_title('R² & Pearson'); axes[2].legend(); axes[2].set_xlabel('Epoch')

    plt.tight_layout()
    plt.savefig(f'{config.output_dir}/training_curves.png', dpi=150)
    plt.show()
    print(f'Best R²     = {max(h["val_r2"]):.4f}  (epoch {h["val_r2"].index(max(h["val_r2"]))+1})')
    print(f'Best MSE    = {min(h["val_mse"]):.4f}  (epoch {h["val_mse"].index(min(h["val_mse"]))+1})')
    print(f'Best Pearson= {max(h["val_pearson"]):.4f}  (epoch {h["val_pearson"].index(max(h["val_pearson"]))+1})')


Best R²     = 0.6724  (epoch 39)
Best MSE    = 0.8692  (epoch 39)
Best Pearson= 0.8219  (epoch 18)


## ✅ CELL 17 — Predicted vs Actual Scatter Plot

In [ ]:
# ── PREDICTED vs ACTUAL SCATTER PLOT ─────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import pearsonr

preds = tm['preds']   # already denormalised in fast_trainer.evaluate()
trues = tm['trues']

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(trues, preds, alpha=0.5, color='steelblue',
           edgecolors='white', s=40, label='Test samples')
mn = min(trues.min(), preds.min()) - 0.2
mx = max(trues.max(), preds.max()) + 0.2
ax.plot([mn, mx], [mn, mx], 'r--', lw=2, label='Perfect prediction')
ax.set_xlabel('True pIC50', fontsize=12)
ax.set_ylabel('Predicted pIC50', fontsize=12)
r2  = 1 - np.sum((trues - preds)**2) / np.sum((trues - trues.mean())**2)
pr, _ = pearsonr(trues, preds)
ax.set_title(f'Predicted vs Actual  |  R²={r2:.3f}  Pearson={pr:.3f}', fontsize=13)
ax.legend()
plt.tight_layout()
plt.savefig(f'{config.output_dir}/scatter_plot.png', dpi=150)
plt.show()


## ✅ CELL 18 — Concept Importance Plot

In [ ]:
# ── CONCEPT IMPORTANCE (from fast_model predictor weights) ────────────────
w = fast_model.pred.predictor[0].weight.detach().cpu().numpy()  # (64, 12)
importance = np.abs(w).mean(axis=0)   # mean abs weight per concept
names = config.concept_names

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['steelblue' if i in config.continuous_concepts else 'coral'
          for i in range(len(names))]
bars = ax.bar(names, importance, color=colors)
ax.set_title('Concept Importance (mean |weight| from predictor)')
ax.set_ylabel('Importance')
plt.xticks(rotation=45, ha='right')

from matplotlib.patches import Patch
ax.legend(handles=[Patch(color='steelblue', label='Continuous'),
                   Patch(color='coral',     label='Binary')], loc='upper right')
plt.tight_layout()
plt.savefig(f'{config.output_dir}/concept_importance.png', dpi=150)
plt.show()


## ✅ CELL 19 — Example Predictions with Explanations

In [ ]:
# ── EXAMPLE PREDICTIONS WITH EXPLANATIONS ────────────────────────────────
import torch
import numpy as np

fast_model.eval()
print('\n' + '='*60)
print('       EXAMPLE PREDICTIONS WITH EXPLANATIONS')
print('='*60)

# Use test_df2 (the split used with fast loaders) if available, else test_df
_tdf = test_df2 if 'test_df2' in dir() else test_df
orig_idx = _tdf.index.tolist()

for i in range(min(5, len(_tdf))):
    row      = _tdf.iloc[i]
    orig     = orig_idx[i]

    drug_emb_t = torch.FloatTensor(drug_embs[orig]).unsqueeze(0).to(device)
    prot_emb_t = torch.FloatTensor(prot_embs_full[orig]).unsqueeze(0).to(device)

    with torch.no_grad():
        py, pc = fast_model(drug_emb_t, prot_emb_t)

    # ── FIX: denormalise prediction back to real pIC50 scale ─────────────
    pred_norm = py.item()
    pred_pic50 = pred_norm * config.pic50_std + config.pic50_mean
    true_pic50 = float(row['pIC50'])
    error      = abs(pred_pic50 - true_pic50)
    concepts   = pc.squeeze().cpu().numpy()

    confidence = ('HIGH binder' if pred_pic50 > 7.5 else
                  'MED binder'  if pred_pic50 > 6.0 else
                  'LOW binder')

    print(f'\n  Sample {i+1}')
    print(f'    True pIC50  : {true_pic50:.3f}')
    print(f'    Predicted   : {pred_pic50:.3f}  (error={error:.3f})')
    print(f'    Confidence  : {confidence}')
    print(f'    Key concepts:')
    for j, name in enumerate(config.concept_names):
        bar = '█' * int(concepts[j] * 20)
        print(f'      {name:<30}: {concepts[j]:.3f}  {bar}')

print('\n' + '='*60)
print(f'Overall test R²={tm["r2"]:.4f}  RMSE={np.sqrt(tm["mse"]):.4f}  Pearson={tm["pearson"]:.4f}')



       EXAMPLE PREDICTIONS WITH EXPLANATIONS

  Sample 1
    True pIC50  : 9.357
    Predicted   : 8.370  (error=0.986)
    Confidence  : HIGH binder
    Key concepts:
      bioavailability               : 0.977  ███████████████████
      metabolic_stability           : 0.044  
      binding_compatibility         : 0.011  
      zinc_binding                  : 0.021  
      hydrogen_bond_donor           : 0.065  █
      hydrogen_bond_acceptor        : 0.815  ████████████████
      hydrophobic_contact           : 0.981  ███████████████████
      steric_clash                  : 0.034  
      molecular_weight_norm         : 0.956  ███████████████████
      logP_norm                     : 0.010  
      rotatable_bonds_norm          : 0.576  ███████████
      aromatic_rings_norm           : 0.988  ███████████████████

  Sample 2
    True pIC50  : 8.328
    Predicted   : 7.297  (error=1.030)
    Confidence  : MED binder
    Key concepts:
      bioavailability               : 0.618  ████████